# Introduccion a brightway - pt. 2

En esta seccion hablaremos de los conceptos fundamentales de brigthway. Es importante aclarar que toda esta informacion esta disponible en linea en la pagina de documentacion:

https://docs.brightway.dev/en/latest/index.html

Este notebook se ejecutara en una nueva sesion, por lo que necesitamos instalar las dependencias una vez mas.

In [ ]:
!pip install -q "bw2calc>=2.1" "bw2data>=4.5" "bw2io>=0.9.11" bw2analyzer polars pypardiso "seaborn>=0.13.2"

<div class="alert alert-block alert-warning">
⚠️ Debes restaurar la sesion!
</div>

Descargamos el respaldo del proyecto, que contiene la base de datos suiza **BAFU-2026** (bases `BAFU-2026` y `BAFU-biosphere`) y los métodos de impacto **IPCC 2021 (GWP100)** y **Ecological Scarcity 2021** (puntos de carga ambiental, UBP).

Los datos BAFU se distribuyen bajo los términos de uso de la Oficina Federal del Medio Ambiente de Suiza (BAFU/OFEV). El archivo está en Google Drive: el dueño debe compartirlo con tu cuenta de Google; inicia sesión con esa cuenta cuando Colab lo pida.

In [ ]:
# Descargamos el respaldo con BAFU-2026 desde Google Drive.
# Inicia sesión con la cuenta de Google con la que se compartió el archivo.
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

FILE_ID = "PEGAR_AQUI_EL_ID_DE_bafu_2026.tar.gz"

auth.authenticate_user()
request = build("drive", "v3").files().get_media(fileId=FILE_ID)
with open("backup.tar.gz", "wb") as fh:
    downloader = MediaIoBaseDownload(fh, request, chunksize=10 * 1024 * 1024)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        print(f"Descargando... {int(status.progress() * 100)}%")

Verificamos

In [ ]:
!du -hs backup.tar.gz

In [4]:
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print

10:13:55-0500

 [

info     

] 

Using environment variable BRIGHTWAY2_DIR for data directory:
C:\Users\ucard\AppData\Local\Temp\claude\35b832068f2ff255\05151dd0-df28-46bc-93ea-7db593b586ec\scratchpad\bwtest

10:13:56-0500

 [

warning  

] 

Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.

### Importar el backup del proyecto
Este modalidad no require mucha explicacion: El proyecto se carga nuevamente.

In [5]:
bi.restore_project_directory(
    'backup.tar.gz',  # nombre del archivo, creado celdas arriba
    project_name='proyecto_bafu', # Se puede elegir un nombre nuevo para el proyecto
    overwrite_existing = False
    )

Restoring project backup archive - this could take a few minutes...

Restored project: proyecto_bafu

'proyecto_bafu'

## Importar bases de datos de inventario
Hemos aprendido a construir un modelo de ACV desde cero y de forma manual. Aunque esto resulta bastante util, en la realidad solemos combinar nuestros datos con aquellos provenientes de bases de datos de inventario. En esta seccion nos enfocaremos en la base de datos **BAFU-2026**, publicada por la Oficina Federal del Medio Ambiente de Suiza (BAFU, por sus siglas en aleman). Es la continuacion de los datos *UVEK LCI* y comparte su origen con ecoinvent v2, por eso los nombres de las actividades se parecen (por ejemplo, `'Rape methyl ester, at esterification plant'`).

En la actualidad hay dos maneras de importar los datos de BAFU en nuestro proyecto:
- Leyendo los archivos ecoSpold v1 crudos directamente del disco y convirtiendolos en una base de datos de brigthway.
- Utilizando la herramienta `bi.remote.install_project`, que descarga un proyecto ya preparado desde un servidor remoto.

### Importando BAFU (crudo) desde el disco

Para este caso, es necesario haber descargado BAFU. BAFU es distribuido en un archivo zip que contiene todas las actividades en formato ecoSpold v1 (un archivo XML por actividad). `bw2io` tiene funciones disenadas para interpretar la informacion, verificar que los `exchanges` sean correctos, y enlazar los flujos elementales con una base de datos de biosfera.

Los archivos de BAFU-2026 necesitan unos pequenos arreglos de formato antes de poder leerlos; el notebook `preparar_base_bafu.ipynb` hace todo el proceso y crea el respaldo que usamos aqui.

In [6]:
# Para importar, hay que seguir los siguentes pasos:
# 1. Leer los archivos XML e dejar que brigthway los interprete.
# db = bi.SingleOutputEcospold1Importer('<carpeta-ecoSpold>', 'BAFU-2026')

In [7]:
# 2. Aplicar una serie de estrategias para asegurarse que no existe informacion corrupta y que la importacion es posible
# db.apply_strategies()
# db.match_database('BAFU-biosphere', fields=('name', 'categories', 'unit'))
# db.statistics()

In [8]:
# 3. BAFU esta listo en la memoria pero aun no ha sido grabado en el disco.
# Hay que grabarlo en el disco.
# db.write_database()

Para verificar que ha sido importado correctamente, podemos repetir el ejercicio realizado con la base de datos de la biosfera de la anterior seccion.

In [9]:
# bd.databases # Lista de las bases de datos

In [10]:
# bafu = bd.Database('BAFU-2026')
# len(bafu) # Muestra la cantidad de elementos

### Importando BAFU desde un servidor remoto
Brightway ofrece algunos proyectos ya preparados. Con la funcion `bi.remote.get_projects()` podemos ver la lista, y con `bi.remote.install_project` instalamos uno. Por ahora solo esta disponible la version anterior, **BAFU-2025-v2**, que trae la base de datos, su biosfera y los metodos Ecological Scarcity 2021.
No requiere usuario ni clave.

In [11]:
# bi.remote.get_projects()
# bi.remote.install_project('BAFU-2025-v2', 'proyecto_bafu_2025')

### Explorando BAFU

In [12]:
import bw2data as bd
bd.projects.set_current("proyecto_bafu")
bd.databases

Databases dictionary with 2 object(s):
	BAFU-2026
	BAFU-biosphere

In [13]:
bafu = bd.Database('BAFU-2026')
print(bafu)

Brightway2 SQLiteBackend: BAFU-2026

In [14]:
len(bafu)

11947

In [15]:
count = 0
for i in bafu:
  if count <10:
    print(i)
    count += 1
  else:
    break

'xxx Sawn timber, softwood, raw, forest-debarked, u=70%, at plant' (cubic meter, RER, ['material, obsolete', 'wood,
obsolete\\products, obsolete'])

'xxx Electricity, low voltage, production CENTREL, at grid' (kilowatt hour, CENTREL, ['electricity', 
'notMaintained'])

'Natural gas, production NL, at long-distance pipeline' (megajoule, RER, ['natural gas', 'transport'])

'Titanium dioxide, chloride process, at plant' (kilogram, RER, ['chemicals', 'inorganic'])

'Hard coal, at regional storage' (kilogram, US, ['fuels', 'coal\\produced coal'])

'Hard coal, burned in power plant' (megajoule, US-FRCC, ['electricity by fuel', 'coal\\power plant'])

'Heavy duty truck, fuel cell electric, 40t gross weight, 2020, urban delivery' (unit, RER, ['transport systems', 
'road\\Transformation\\Infrastructure'])

'Natural gas, production AU, at evaporation plant' (normal cubic meter, JP, ['fuels', 'natural gas\\produced gas'])

'Carbon dioxide, captured at municipal solid waste incineration plant, for subsequent reuse' (kilogram, CH, 
['chemicals', 'gases\\transformation'])

'Disposal, displacement pile, concrete 620mm, high reinforced, Integra Wohnen Wallisellen' (meter, CH, 
['construction processes', 'civil engineering'])

In [16]:
seleccionado = bafu.random() # Explora las actividades
# Primero exploramos los keys de la actividad seleccionada
print(list(seleccionado.keys()))

[
    'tags',
    'references',
    'categories',
    'code',
    'comment',
    'comments',
    'authors',
    'database',
    'filename',
    'location',
    'name',
    'unit',
    'type',
    'reference product',
    'production amount',
    'id'
]

In [17]:
# Podemos ver el contenido de todo el dataset.
print(seleccionado.as_dict())

{
    'tags': [
        ('ecoSpold01datasetRelatesToProduct', True),
        ('ecoSpold01infrastructureProcess', False),
        ('ecoSpold01infrastructureIncluded', False),
        ('ecoSpold01localName', 'Crude oil, production US, at long distance transport'),
        ('ecoSpold01localCategory', 'fuels'),
        ('ecoSpold01localSubCategory', 'oil\\transport'),
        ('ecoSpold01category', 'fuels'),
        ('ecoSpold01subCategory', 'oil\\transport'),
        (
            'ecoSpold01includedProcesses',
            'Transportation of crude oil from exploration site to refinery in Switzerland. Includes transport 
service requirements and emissions from oil handling and evaporation.'
        ),
        ('ecoSpold01dataValidForEntirePeriod', True),
        ('ecoSpold01endDate', '2015-11-11'),
        ('ecoSpold01startDate', '1992-11-11'),
        ('ecoSpold01type', 1),
        ('ecoSpold01impactAssessmentResult', False),
        ('ecoSpold01version', '2023'),
        ('ecoSpold01internalVersion', '0.0'),
        ('ecoSpold01timestamp', '2018-10-22T00:00:00+02:00'),
        ('ecoSpold01languageCode', 'en'),
        ('ecoSpold01localLanguageCode', 'de'),
        ('ecoSpold01energyValues', 0)
    ],
    'references': [
        {
            'identifier': 149975,
            'type': 'Seperate publication',
            'authors': ['Meili C.', 'Jungbluth N., Bussa M.'],
            'year': 2021,
            'title': '2021 - LCI long-dist. transp. crude oil - Meili',
            'pages': '',
            'editors': '-',
            'anthology': 'Project report',
            'place_of_publication': 'Schaffhausen, CH',
            'publisher': 'ESU-services Ltd.',
            'journal': '',
            'volume': 565,
            'issue': '',
            'text': 'Meili C., Jungbluth N., Wenzel P. (2018) Life cycle inventory for long distance transport of 
crude oil. Project report. Volume: 565. ESU-services Ltd.. Schaffhausen, CH.\\nType: Separate publication\\nSource:
Report\\nFirst author: Meili C.\\nOther authors: Jungbluth N., Wenzel P.\\nYear: 2018\\nTitle of anthology: Project
report\\nPlace of publication: Schaffhausen, CH\\nPublisher: ESU-services Ltd.\\nVolume No: 565\\n'
        }
    ],
    'categories': ['fuels', 'oil\\transport'],
    'code': '26e3c705c5b40a74edd3762954410a25',
    'comment': 'Calculation for transport distances assuming transport by pipeline offshore and onshore as well as 
sea transport in tanker. Sites and modes of transportation based on the supply situation in 2016.;\nUUID: 
5188a405-6e97-3a3b-9793-b93af078682b\nTransportation of crude oil from exploration site to refinery in Switzerland.
Includes transport service requirements and emissions from oil handling and evaporation.\nLocation: Calculations 
based on assumptions for supply to Swiss refineries.\nTechnology: Operation of crude oil pipelines by 
electricity.\nTime period: Transport modes investigated for 2016\nProduction volume: 490k tons of directly imported
crude oil in CH in 2016\nSampling: Literature. Online calculators for distances. \nExtrapolations: 
none\nUncertainty adjustments: none',
    'comments': {
        'generalComment': 'Calculation for transport distances assuming transport by pipeline offshore and onshore 
as well as sea transport in tanker. Sites and modes of transportation based on the supply situation in 
2016.;\nUUID: 5188a405-6e97-3a3b-9793-b93af078682b',
        'includedProcesses': 'Transportation of crude oil from exploration site to refinery in Switzerland. 
Includes transport service requirements and emissions from oil handling and evaporation.',
        'location': 'Location: Calculations based on assumptions for supply to Swiss refineries.',
        'technology': 'Technology: Operation of crude oil pipelines by electricity.',
        'timePeriod': 'Time period: Transport modes investigated for 2016',
        'productionVolume': 'Production volume: 490k tons of directly imported crude oil in CH in 2016',
        's

Como pueden notar, el contenido de la actividad BAFU es bastante rica. Existen campos fuera de `name`, `code`,`location` y `unit` que son nuevos para nosotros, lo que demuestra que brightway es lo suficientemente flexible al definir una actividad.

Lo que vimos en la celda anterior describe a una actividad, pero aun no describe sus conexiones (`exchanges`). Para acceder a ellas, hay que utilizar las funciones `exchanges`, `technosphere` o `biosphere`, segun lo que se desee observar.

In [18]:
# `exchanges` retorna un objeto the brightway que no es nativo de python
type(seleccionado.exchanges())

bw2data.backends.proxies.Exchanges

In [19]:
# Si deseamos leerlo al estilo de una lista, hay que convertirlo en una lista.
for exchange in seleccionado.exchanges():
    print(exchange)
# print(list(seleccionado.exchanges()))

Exchange: 1.0 kilogram 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 
'oil\\transport']) to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 
'oil\\transport'])

Exchange: 4.5e-07 kilogram 'Benzene' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at long 
distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 5.3876e-05 kilogram 'Hexane' (kilogram, None, ('emissions to air', 'high. pop.')) to 'Crude oil, 
production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 2.7e-07 kilogram 'Toluene' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at long 
distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 0.00012642 kilogram 'Butane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 10.1 ton kilometer 'Transport, transoceanic tanker' (ton kilometer, OCE, ['transport systems', 'ship']) 
to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

... (4 outputs left out)

Exchange: 1.498e-06 kilogram 'Methane, fossil' (kilogram, None, ('emissions to air',)) to 'Crude oil, production 
US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 9.558e-05 kilogram 'Pentane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 6.1296e-05 kilogram 'Propane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.72 ton kilometer 'Transport, crude oil pipeline, onshore' (ton kilometer, RER, ['transport systems', 
'pipeline']) to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

In [20]:
# Si deseamos solo la tecnosfera, usamos la funcion correspondiente
for exchange in seleccionado.technosphere():
    print(exchange)
# print(list(seleccionado.technosphere()))

Exchange: 10.1 ton kilometer 'Transport, transoceanic tanker' (ton kilometer, OCE, ['transport systems', 'ship']) 
to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 0.15 ton kilometer 'Transport, crude oil pipeline, offshore' (ton kilometer, OCE, ['transport systems', 
'pipeline']) to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.0003 kilogram 'Crude oil, at production' (kilogram, US, ['oil', 'production']) to 'Crude oil, 
production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.72 ton kilometer 'Transport, crude oil pipeline, onshore' (ton kilometer, RER, ['transport systems', 
'pipeline']) to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

In [21]:
for exchange in seleccionado.biosphere():
    print(exchange)

Exchange: 4.5e-07 kilogram 'Benzene' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at long 
distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 5.3876e-05 kilogram 'Hexane' (kilogram, None, ('emissions to air', 'high. pop.')) to 'Crude oil, 
production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 2.7e-07 kilogram 'Toluene' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at long 
distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 0.00012642 kilogram 'Butane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.601e-06 kilogram 'Ethane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at long
distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.35e-06 kilogram 'Hydrocarbons, aliphatic, alkanes, unspecified' (kilogram, None, ('emissions to air',))
to 'Crude oil, production US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 1.498e-06 kilogram 'Methane, fossil' (kilogram, None, ('emissions to air',)) to 'Crude oil, production 
US, at long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 9.558e-05 kilogram 'Pentane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

Exchange: 6.1296e-05 kilogram 'Propane' (kilogram, None, ('emissions to air',)) to 'Crude oil, production US, at 
long distance transport' (kilogram, CH, ['fuels', 'oil\\transport'])

La impresion realizada en la celda de arriba nos muestra la informacion necesaria para poder construir las matrices. Sin embargo, brightway nos permite manipular el `exchange` y acceder a su metadata.

In [22]:
# Seleccionamos el primer `exchange` de la tecnosfera
exchange = list(seleccionado.technosphere())[0]
print(exchange.as_dict())

{
    'categories': ('transport systems', 'ship'),
    'location': 'OCE',
    'unit': 'ton kilometer',
    'name': 'Transport, transoceanic tanker',
    'type': 'technosphere',
    'infrastructureProcess': False,
    'comment': '(4,1,1,3,1,na); Calculation 2016\n',
    'uncertainty type': 2,
    'amount': 10.1,
    'loc': np.float64(2.312535423847214),
    'scale': 0.3585050904122363,
    'negative': False,
    'input': ('BAFU-2026', '7bff046b28f9ce688d0faaf1a5d34cbc'),
    'output': ('BAFU-2026', '26e3c705c5b40a74edd3762954410a25')
}

## Opciones de busqueda
Como podran imaginar, manipular una base de datos con tantas actividades (~12k) es bastante complicado. Podemos utilizar funciones nativas de python para realizar una busqueda.

In [23]:
for x in bafu:
  if x['name'] == 'Transport, freight, lorry, 32t-40t gross weight, fleet average':
    print(x)

'Transport, freight, lorry, 32t-40t gross weight, fleet average' (ton kilometer, RER, ['transport systems', 
'road\\Transformation'])

'Transport, freight, lorry, 32t-40t gross weight, fleet average' (ton kilometer, CH, ['transport systems', 
'road\\Transformation'])

In [24]:
for act in bafu:
  if 'lorry' in act['name'] and 'RER' in act['location']:
    print(act)

'Transport, freight, lorry, diesel, 26t gross weight, fleet average, long haul' (ton kilometer, RER, ['transport 
systems', 'road\\Transformation'])

'Transport, freight, lorry, diesel, 32t gross weight, 2020, EURO-VI, urban delivery' (ton kilometer, RER, 
['transport systems', 'road\\Transformation'])

'Transport, freight, lorry, diesel, 40t gross weight, 2010, EURO-V, long haul' (ton kilometer, RER, ['transport 
systems', 'road\\Transformation'])

'Power electronics, for lorry' (kilogram, RER, ['transport systems', 'road\\Vehicle components'])

'Transport, freight, lorry, diesel, 3.5t gross weight, 2002, EURO-III, long haul' (ton kilometer, RER, ['transport 
systems', 'road\\Transformation'])

'Transport, freight, lorry, 3.5t-7.5t gross weight, fleet average' (ton kilometer, RER, ['transport systems', 
'road\\Transformation'])

... (97 outputs left out)

'Transport, freight, lorry, diesel, 3.5t gross weight, 2006, EURO-IV, regional delivery' (ton kilometer, RER, 
['transport systems', 'road\\Transformation'])

'Transport, freight, lorry, diesel, 3.5t gross weight, 2006, EURO-IV, urban delivery' (ton kilometer, RER, 
['transport systems', 'road\\Transformation'])

'Transport, freight, lorry, diesel, 26t gross weight, 2020, EURO-VI, regional delivery' (ton kilometer, RER, 
['transport systems', 'road\\Transformation'])

'Transport, freight, lorry, diesel, 26t gross weight, 2010, EURO-V, long haul' (ton kilometer, RER, ['transport 
systems', 'road\\Transformation'])

In [25]:
# camion = [x for x in bafu if x['name'] == 'Transport, freight, lorry, 32t-40t gross weight, fleet average'][0]
# camion

Esta manera de buscar es mas 'pythonic'. Sin embargo, tambien puedes usar el buscador de brightway a traves de la funcion `search`.

In [26]:
bafu.search('transport freight lorry 32t-40t CH')

['Transport, freight, lorry, 32t-40t gross weight, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation'])]

In [27]:
help(bafu.search)  # La funcion search prioriza algunos campos para hacer el filtro.

Help on method search in module bw2data.backends.base:

search(string, **kwargs) method of bw2data.backends.base.SQLiteBackend instance
    Search this database for ``string``.
    
    The searcher include the following fields:
    
    * name
    * comment
    * categories
    * location
    * reference product
    
    ``string`` can include wild cards, e.g. ``"trans*"``.
    
    By default, the ``name`` field is given the most weight. The full weighting set is called the ``boost`` dictionary, and the default weights are::
    
        {
            "name": 5,
            "comment": 1,
            "product": 3,
            "categories": 2,
            "location": 3
        }
    
    Optional keyword arguments:
    
    * ``limit``: Number of results to return.
    * ``boosts``: Dictionary of field names and numeric boosts - see default boost values above. New values must be in the same format, but with different weights.
    * ``filter``: Dictionary of criteria that search results

# Introduccion a brightway - pt. 3

En esta seccion hablaremos de los conceptos fundamentales de brigthway. Es importante aclarar que toda esta informacion esta disponible en linea en la pagina de documentacion:

https://docs.brightway.dev/en/latest/index.html

## Explorando las matrices
Ahora que sabemos como crear un actividad y metodos desde cero. Podemos concentrarnos en manipular las actividades que estan presentes en BAFU.
Para esta parte usaremos un proyecto que hemos preparado para ustedes que contiene una biosfera y tecnosfera de BAFU-2026

In [28]:
# Re-import for this section if running independently
import bw2data as bd
import bw2io as bi
import bw2calc as bc
from rich import print

Tenemos dos bases de datos

In [29]:
bd.databases

Databases dictionary with 2 object(s):
	BAFU-2026
	BAFU-biosphere

In [30]:
bafu.search('rape methyl ester CH')

['Rape methyl ester, at esterification plant' (kilogram, CH, ['fuels', 'biofuels']),
 'Rape methyl ester, at regional storage' (kilogram, CH, ['fuels', 'biofuels']),
 'Operation, lorry 28t, rape methyl ester 100%' (kilometer, CH, ['transport systems', 'road\\operations']),
 'Operation, passenger car, rape seed methyl ester 5%' (kilometer, CH, ['transport systems', 'road\\operations']),
 'Rape methyl ester, production RER, at service station' (kilogram, CH, ['fuels', 'biofuels']),
 'Glycerine, from rape oil, at esterification plant' (kilogram, CH, ['chemicals', 'organic'])]

In [31]:
# seleccionamos la base de datos BAFU y una actividad que tomaremos de ejemplo:
# 1 kg de biodiesel de colza (rape methyl ester) producido en Suiza
bafu = bd.Database("BAFU-2026")
biodiesel = [a for a in bafu.search('rape methyl ester esterification plant CH')
             if a['name'] == 'Rape methyl ester, at esterification plant' and a['location'] == 'CH'][0]
biodiesel

'Rape methyl ester, at esterification plant' (kilogram, CH, ['fuels', 'biofuels'])

In [32]:
bd.methods

Methods dictionary with 22 objects, including:
	('Ecological Scarcity 2021', 'Biotic Resources')
	('Ecological Scarcity 2021', 'Carcinogenic substances into air')
	('Ecological Scarcity 2021', 'Energy resources')
	('Ecological Scarcity 2021', 'Global warming')
	('Ecological Scarcity 2021', 'Heavy metals into air')
	('Ecological Scarcity 2021', 'Heavy metals into soil')
	('Ecological Scarcity 2021', 'Heavy metals into water')
	('Ecological Scarcity 2021', 'Land use')
	('Ecological Scarcity 2021', 'Main air pollutants and PM')
	('Ecological Scarcity 2021', 'Mineral resources')
Use `list(this object)` to get the complete list.

In [33]:
print(list(bd.methods))

[
    ('Ecological Scarcity 2021', 'Water resources, evaporated'),
    ('Ecological Scarcity 2021', 'Water resources, net balance'),
    ('Ecological Scarcity 2021', 'Energy resources'),
    ('Ecological Scarcity 2021', 'Mineral resources'),
    ('Ecological Scarcity 2021', 'Land use'),
    ('Ecological Scarcity 2021', 'Global warming'),
    ('Ecological Scarcity 2021', 'Ozone layer depletion'),
    ('Ecological Scarcity 2021', 'Main air pollutants and PM'),
    ('Ecological Scarcity 2021', 'Carcinogenic substances into air'),
    ('Ecological Scarcity 2021', 'Heavy metals into air'),
    ('Ecological Scarcity 2021', 'Water pollutants'),
    ('Ecological Scarcity 2021', 'POP into water'),
    ('Ecological Scarcity 2021', 'Heavy metals into water'),
    ('Ecological Scarcity 2021', 'Pesticides into soil'),
    ('Ecological Scarcity 2021', 'Heavy metals into soil'),
    ('Ecological Scarcity 2021', 'Radioactive substances into air'),
    ('Ecological Scarcity 2021', 'Radioactive substances into water'),
    ('Ecological Scarcity 2021', 'Noise'),
    ('Ecological Scarcity 2021', 'Waste, non radioactive'),
    ('Ecological Scarcity 2021', 'Radioactive waste to deposit'),
    ('Ecological Scarcity 2021', 'Biotic Resources'),
    ('IPCC 2021', 'climate change', 'global warming potential (GWP100)')
]

In [34]:
for method in bd.methods:
  if 'IPCC 2021' in method[0] and "GWP100" in method[2]:
    print(method)

('IPCC 2021', 'climate change', 'global warming potential (GWP100)')

In [35]:
# Elegimos un metodo que ya esta instalado y hace un LCA pero nos detenemos en la etapa de LCI
method=('IPCC 2021', 'climate change', 'global warming potential (GWP100)')
lca = bc.LCA({biodiesel:1},method=method) # Instancia la clase
lca.lci() # calcula el inventario de ciclo de vida

Recordemos que BAFU-2026 tiene 11947 actividades
Entonces que dimensiones deberia tener la matriz de la tecnosfera?

In [36]:
lca.technosphere_matrix.toarray().shape

(11947, 11947)

Que dimensiones deberia tener el vector s?

In [37]:
lca.supply_array

array([1.08809572e-08, 0.00000000e+00, 3.96421085e-06, ...,
       1.03244473e-05, 2.54409638e-13, 0.00000000e+00], shape=(11947,))

Si quisiera saber cuanta 'semilla de colza (produccion integrada, IP) en finca'
se requiere en TOTAL para producir 1 kg de biodiesel...

In [38]:
bafu.search('rape seed IP at farm')

['Rape seed IP, at farm' (kilogram, CH, ['agricultural', 'plant production']),
 'Rape seed IP, at regional storehouse' (kilogram, CH, ['agricultural', 'plant production\\seeds'])]

In [39]:
colza = [a for a in bafu.search('rape seed IP at farm')
         if a['name'] == 'Rape seed IP, at farm' and a['location'] == 'CH'][0]
colza

'Rape seed IP, at farm' (kilogram, CH, ['agricultural', 'plant production'])

In [40]:
colza.id

378785761555673088

In [41]:
# el lca.activity_dict me permite ubicar una actividad en la matriz.
lca.supply_array[lca.activity_dict[colza.id]]

np.float64(1.577789853709733)

Ahora continuamos con el LCIA

In [42]:
lca.lcia() # Calcula los impactos
print("El impacto es: ", lca.score)

El impacto es:  2.2468106051983776

## Analisis de contribuciones
Para entender las distintas contribuciones, tenemos que seguir utilizando el objeto LCA.
Este objeto mantiene los resultados del ACV en memoria

### Procesos mas importantes
Para listar los procesos que generan mas impactos utilizaremos el paquete `bw2analyzer` y `pandas`.

In [43]:
import pandas as pd
import bw2analyzer as ba
ba.ContributionAnalysis.annotated_top_processes??

In [44]:
ba.ContributionAnalysis().annotated_top_processes(lca=lca) # dificil de visulizar

[(np.float64(0.6213382254661363),
  np.float64(1.577789853709733),
  'Rape seed IP, at farm' (kilogram, CH, ['agricultural', 'plant production'])),
 (np.float64(0.2860524700772981),
  np.float64(0.6194262604738525),
  'Rape seed extensive, at farm' (kilogram, CH, ['agricultural', 'plant production'])),
 (np.float64(0.27332715362656),
  np.float64(0.11933234257356184),
  'Nitric acid, 50% in H2O, at plant' (kilogram, RER, ['chemicals', 'acids (inorganic)'])),
 (np.float64(0.11541467513541875),
  np.float64(0.07880637609968578),
  'Ammonia, steam reforming, liquid, at plant' (kilogram, RER, ['chemicals', 'inorganic'])),
 (np.float64(0.09661438274480522),
  np.float64(0.005537593211148015),
  'Natural gas, vented' (normal cubic meter, GLO, ['oil', 'production'])),
 (np.float64(0.07850671667449882),
  np.float64(1.0627884127838834),
  'Light fuel oil, burned in industrial furnace 1MW, non-modulating' (megajoule, CH, ['heat', 'oil\\heating systems'])),
 (np.float64(0.07804292147249757),
  n

In [ ]:
contrib = pd.DataFrame(
    [(x, y, z["unit"], z["name"], z["location"]) for x, y, z in ba.ContributionAnalysis().annotated_top_processes(lca=lca)],
    columns=["score", "quantity", "unit", "name", "location"]
)
# score está en kg CO2-eq; para verlo como fracción del total lo dividimos por lca.score
contrib.insert(1, "fraccion", contrib["score"] / lca.score)
print("Suma de las fracciones de estos 25 procesos:", contrib["fraccion"].sum())
contrib

**Cómo leer la tabla:**
- `score` es el impacto (kg CO2-eq) de las emisiones **directas** de cada proceso, no de toda su cadena de suministro. Por eso aparecen procesos como el ácido nítrico, que emite N2O directamente.
- `quantity` es cuánto de ese proceso se necesita en total para 1 kg de biodiesel, **en su propia unidad** (`unit`). Por ejemplo, 1.58 kg de semilla de colza, pero 0.00075 hectáreas de cosecha con cosechadora.
- `score` está en kg CO2-eq, no es una fracción: sumando **todos** los procesos se obtiene `lca.score` (2.25 kg CO2-eq), no 1. La columna `fraccion` = `score / lca.score` es la participación de cada proceso en el total.
- La tabla muestra solo los 25 procesos más importantes, que suman cerca de 0.87 (87%) del total; el 13% restante viene de miles de procesos con contribuciones pequeñas.

In [ ]:
my_data = pd.DataFrame(
    [(x, y, z["unit"], z["name"], z["location"]) for x, y, z in ba.ContributionAnalysis().annotated_top_processes(lca=lca)],
    columns=["score", "quantity", "unit", "name", "location"]
)
my_data.to_csv('my_export.csv')

### Emisiones mas importantes
De manera similar, podemos obtener el ranking de flujos ambiental que generan mayores impactos

In [47]:
import pandas as pd
import bw2analyzer as ba
pd.DataFrame(
    [(x, y, z["name"], z["categories"]) for x, y, z in ba.ContributionAnalysis().annotated_top_emissions(lca=lca)],
    columns=["score", "quantity", "name", "categories"]
)

,score,quantity,name,categories
0,0.899910,3.296372e-03,Dinitrogen monoxide,"(emissions to air, low. pop.)"
1,0.448303,4.483033e-01,"Carbon dioxide, fossil","(emissions to air, high. pop.)"
2,0.355841,3.558408e-01,"Carbon dioxide, fossil","(emissions to air, low. pop.)"
3,0.274780,1.006522e-03,Dinitrogen monoxide,"(emissions to air, high. pop.)"
4,0.135142,1.351421e-01,"Carbon dioxide, fossil","(emissions to air,)"
5,0.098133,3.293070e-03,"Methane, fossil","(emissions to air,)"
6,0.026537,8.905085e-04,"Methane, fossil","(emissions to air, low. pop.)"
7,0.001814,6.087485e-05,"Methane, fossil","(emissions to air, high. pop.)"
8,0.001283,4.752241e-05,"Methane, biogenic","(emissions to air, high. pop.)"
9,0.001277,5.067080e-08,Sulfur hexafluoride,"(emissions to air,)"


La importancia de las emisiones en el impacto tiene que ver con la cantidad y con los factores de caracterizacion.
Podemos listar estos factores para revisarlos

In [48]:
for flow_id, cf in bd.Method(method).load():
    # En bw2data 4, load() devuelve el id numérico de cada flujo
    print(bd.get_node(id=flow_id), cf)

'Carbon dioxide, fossil' (kilogram, None, ('emissions to air',)) 1.0

'Carbon dioxide, fossil' (kilogram, None, ('emissions to air', 'stratosphere + troposphere')) 1.0

'Carbon dioxide' (kilogram, None, ('emissions to air', 'low. pop.')) 1.0

'Pentane, perfluoro-' (kilogram, None, ('emissions to air',)) 9220.0

'Methane, dichlorofluoro-, HCFC-21' (kilogram, None, ('emissions to air',)) 160.0

'Carbon dioxide, land transformation' (kilogram, None, ('emissions to air', 'low. pop.')) 1.0

... (92 outputs left out)

'Ethane, 1,1,1,2-tetrafluoro-, HFC-134a' (kilogram, None, ('emissions to air', 'low. pop.')) 1530.0

'Ethane, 1,1-difluoro-, HFC-152a' (kilogram, None, ('emissions to air', 'low. pop.')) 164.0

'Methane, fossil' (kilogram, None, ('emissions to air', 'high. pop.')) 29.8

'Methane, difluoro-, HFC-32' (kilogram, None, ('emissions to air', 'high. pop.')) 771.0

In [49]:
bd.Method(method).load()

[(378785576649781249, 1.0),
 (378785575689285633, 1.0),
 (378785576192602124, 1.0),
 (378785576511369225, 9220.0),
 (378785576016441350, 160.0),
 (378785576083550211, 1.0),
 (378785576721084421, 1930.0),
 (378785576108716051, 11.2),
 (378785576725278721, 7200.0),
 (378785576444260359, 29.8),
 (378785575961915395, 1960.0),
 (378785576251322376, 12500.0),
 (378785576461037572, 1530.0),
 (378785576049995777, 1.0),
 (378785576242933765, 160.0),
 (378785576469426183, 20.6),
 (378785576234545154, 12400.0),
 (378785576175824898, 11.2),
 (378785576725278725, 27.0),
 (378785576788193283, 27.0),
 (378785575789948942, 25200.0),
 (378785576205185030, 1960.0),
 (378785576435871744, 161.0),
 (378785576079355908, 164.0),
 (378785575940943879, 0.006),
 (378785575605399553, 29.8),
 (378785576297459716, 7200.0),
 (378785576284876804, 1.0),
 (378785576444260363, 27.0),
 (378785576825942022, 6230.0),
 (378785575861252104, 20.6),
 (378785575588622340, 14600.0),
 (378785576091938832, 6.34),
 (37878557612129

## Metodo suizo: Ecological Scarcity 2021
BAFU usa como metodo principal **Ecological Scarcity 2021** (metodo de la escasez ecologica). Cada categoria se expresa en *puntos de carga ambiental* (UBP, *Umweltbelastungspunkte*), y la suma de todas las categorias da un unico puntaje.
Podemos reutilizar el mismo objeto `lca` y cambiar solo el metodo con `switch_method`.

*Nota:* el resultado es muy cercano al publicado por BAFU (archivo *LCIA Results*), pero no identico: esta version del metodo no incluye factores para la categoria *Noise* (ruido), y *Energy resources* resulta algo menor. Para 1 kg de biodiesel la diferencia en el total es de menos de 1%.

In [50]:
resultados = []
for m in bd.methods:
    if m[0] == 'Ecological Scarcity 2021' and bd.methods[m].get("num_cfs", 1) > 0:
        lca.switch_method(m)
        lca.lcia()
        resultados.append((m[1], lca.score))

es2021 = pd.DataFrame(resultados, columns=["categoria", "UBP"]).sort_values("UBP", ascending=False)
print("Total:", es2021["UBP"].sum(), "UBP por kg de biodiesel")
es2021

Total: 16088.961471190303 UBP por kg de biodiesel

,categoria,UBP
13,Pesticides into soil,4848.189586
4,Land use,3563.438102
10,Water pollutants,3413.203584
5,Global warming,2238.024091
7,Main air pollutants and PM,913.988214
14,Heavy metals into soil,581.052255
2,Energy resources,259.097089
18,Radioactive waste to deposit,120.274951
9,Heavy metals into air,49.084236
17,"Waste, non radioactive",19.977178
